In [0]:
tabelas = [
    "zeta_project.gold.dim_customers",
    "zeta_project.gold.dim_products",
    "zeta_project.gold.fact_orders",
    "zeta_project.gold.fact_order_items"
]

# Caminho do seu Volume
pasta_final = "/Volumes/zeta_project/bronze/arquivos"
pasta_temp = f"{pasta_final}/temp_extracao"

# Cria a pasta final no Volume (caso ainda não exista)
dbutils.fs.mkdirs(pasta_final)

print(f"Iniciando a extração para o Volume: {pasta_final}\n")

for tabela in tabelas:
    # Extrai as partes do nome da tabela (ex: ['zeta_project', 'bronze', 'customers'])
    partes = tabela.split('.')
    schema = partes[1]
    nome_tabela = partes[2]
    
    # Monta o nome no padrão solicitado: schema.nometabela
    nome_arquivo = f"{schema}.{nome_tabela}"
    
    caminho_temp_tabela = f"{pasta_temp}/{nome_arquivo}"
    arquivo_final = f"{pasta_final}/{nome_arquivo}.csv"
    
    print(f"⏳ Processando: {nome_arquivo}...")
    
    # 1. Salva o CSV temporário usando Spark (para garantir performance e não estourar memória)
    df = spark.table(tabela)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(caminho_temp_tabela)
    
    # 2. Localiza o arquivo 'part-0000...' gerado pelo Spark
    arquivos_gerados = dbutils.fs.ls(caminho_temp_tabela)
    arquivo_part = [arq.path for arq in arquivos_gerados if arq.name.startswith("part-") and arq.name.endswith(".csv")][0]
    
    # 3. Copia e renomeia o arquivo para a raiz do seu Volume
    dbutils.fs.cp(arquivo_part, arquivo_final)
    print(f"   ✅ Salvo como: {nome_arquivo}.csv")

# 4. Remove a pasta temporária de sujeira do Spark
dbutils.fs.rm(pasta_temp, recurse=True)

print("\n🚀 Extração finalizada com sucesso!")
print(f"Seus arquivos estão prontos no caminho: {pasta_final}")